In [0]:
%pip install faker

In [0]:
import random
import uuid
from datetime import datetime, timedelta
 
import pandas as pd
from faker import Faker
 
fake = Faker("en_GB")   # en_GB gives UK-style names, addresses, postcodes
Faker.seed(42)
random.seed(42)
 
print("Imports done. Seed set — data will be identical on every run.")

In [0]:
CONFIG = {
    "n_customers":       50_000,
    "n_accounts":        55_000,    # slightly more than customers — some have >1 account
    "n_transactions":    200_000,   # high volume to make DQ rules statistically meaningful
 
    # Dirt rates — what % of records get each type of intentional error
    "null_rate":         0.03,      # 3% of optional fields go NULL
    "duplicate_rate":    0.01,      # 1% of rows are duplicated
    "future_date_pct":   0.005,     # 0.5% of transaction dates are in the future
    "invalid_email_pct": 0.02,      # 2% of emails are malformed
 
    # Unity Catalog Volume path — raw CSVs land here
    "volume_path": "/Volumes/workspace/banking_datasentry/datasentry_files",
}
 
print("Config loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [0]:
def inject_nulls(series: pd.Series, rate: float) -> pd.Series:
    """Replace `rate` fraction of values in a Series with None."""
    mask = pd.Series(
        [random.random() < rate for _ in range(len(series))],
        index=series.index
    )
    return series.where(~mask, other=None)

In [0]:
print("Generating customers...")
 
n = CONFIG["n_customers"]
customer_ids = [str(uuid.uuid4()) for _ in range(n)]
 
customers = pd.DataFrame({
    "customer_id":   customer_ids,
    "full_name":     [fake.name() for _ in range(n)],
    "email":         [fake.email() for _ in range(n)],
    "phone":         [fake.phone_number() for _ in range(n)],
    "date_of_birth": [
        fake.date_of_birth(minimum_age=18, maximum_age=80).isoformat()
        for _ in range(n)
    ],
    "country": [
        random.choice(["IE"] * 85 + ["GB", "US", "DE", "FR", "IN"] * 3)
        for _ in range(n)
    ],
    "created_at": [
        fake.date_time_between(start_date="-5y", end_date="now").isoformat()
        for _ in range(n)
    ],
})
 
# Inject malformed emails (missing @ symbol)
bad_email_idx = customers.sample(frac=CONFIG["invalid_email_pct"]).index
customers.loc[bad_email_idx, "email"] = customers.loc[bad_email_idx, "email"].str.replace("@", "", regex=False)
 
# Inject nulls into optional fields
customers["phone"]         = inject_nulls(customers["phone"],         CONFIG["null_rate"])
customers["date_of_birth"] = inject_nulls(customers["date_of_birth"], CONFIG["null_rate"])
 
# Inject duplicate rows
n_dupes = int(n * CONFIG["duplicate_rate"])
dupes = customers.sample(n=n_dupes, replace=True)
customers = pd.concat([customers, dupes], ignore_index=True)
 
print(f"  Rows (incl. dupes) : {len(customers):,}")
print(f"  Null phones        : {customers['phone'].isna().sum():,}")
print(f"  Malformed emails   : {customers['email'].str.contains('@').eq(False).sum():,}")

In [0]:
print("Generating accounts...")
 
n = CONFIG["n_accounts"]
 
def pick_customer_id(valid_ids, orphan_rate=0.02):
    if random.random() < orphan_rate:
        return str(uuid.uuid4())   # orphan — FK violation
    return random.choice(valid_ids)
 
account_customer_ids = [pick_customer_id(customer_ids) for _ in range(n)]
account_types = [random.choice(["CURRENT", "SAVINGS", "LOAN"]) for _ in range(n)]
 
balances = []
for atype in account_types:
    if atype == "CURRENT":
        balances.append(round(random.uniform(-500, 50_000), 2))
    elif atype == "SAVINGS":
        balances.append(round(random.uniform(0, 100_000), 2))
    else:
        balances.append(round(random.uniform(1_000, 50_000), 2))
 
accounts = pd.DataFrame({
    "account_id":   [str(uuid.uuid4()) for _ in range(n)],
    "customer_id":  account_customer_ids,
    "account_type": account_types,
    "status":       [random.choice(["ACTIVE", "ACTIVE", "ACTIVE", "INACTIVE", "CLOSED"]) for _ in range(n)],
    "balance":      balances,
    "currency":     [random.choice(["EUR"] * 80 + ["GBP"] * 15 + ["USD"] * 5) for _ in range(n)],
    "opened_date":  [
        fake.date_between(start_date="-10y", end_date="today").isoformat()
        for _ in range(n)
    ],
})
 
accounts["balance"] = inject_nulls(accounts["balance"], CONFIG["null_rate"])
 
print(f"  Rows           : {len(accounts):,}")
print(f"  Null balances  : {accounts['balance'].isna().sum():,}")
print(f"  Orphaned FKs   : ~{int(n * 0.02):,} (approx)")

In [0]:
print("Generating transactions...")
 
n = CONFIG["n_transactions"]
account_id_list = accounts["account_id"].tolist()
 
txn_account_ids = [
    pick_customer_id(account_id_list, orphan_rate=0.02)
    for _ in range(n)
]
 
today = datetime.today()
 
def gen_txn_date(future_rate=CONFIG["future_date_pct"]):
    if random.random() < future_rate:
        return (today + timedelta(days=random.randint(1, 30))).isoformat()
    return fake.date_time_between(start_date="-3y", end_date="now").isoformat()
 
def gen_amount():
    if random.random() < 0.01:
        return 0.00
    return round(random.uniform(0.01, 10_000), 2)
 
transactions = pd.DataFrame({
    "transaction_id":   [str(uuid.uuid4()) for _ in range(n)],
    "account_id":       txn_account_ids,
    "transaction_type": [random.choice(["DEBIT", "CREDIT", "TRANSFER"]) for _ in range(n)],
    "amount":           [gen_amount() for _ in range(n)],
    "currency":         [random.choice(["EUR"] * 78 + ["GBP"] * 15 + ["USD"] * 5 + ["XYZ"] * 2) for _ in range(n)],
    "transaction_date": [gen_txn_date() for _ in range(n)],
    "merchant":         [fake.company() for _ in range(n)],
    "status":           [
        random.choice(["COMPLETED"] * 80 + ["PENDING"] * 10 + ["FAILED"] * 7 + ["REVERSED"] * 3)
        for _ in range(n)
    ],
})
 
transactions["merchant"] = inject_nulls(transactions["merchant"], CONFIG["null_rate"])
 
n_dupes = int(n * CONFIG["duplicate_rate"])
dupes = transactions.sample(n=n_dupes, replace=True)
transactions = pd.concat([transactions, dupes], ignore_index=True)
 
print(f"  Rows (incl. dupes) : {len(transactions):,}")
print(f"  Zero-amount rows   : {(transactions['amount'] == 0).sum():,}")
print(f"  Null merchants     : {transactions['merchant'].isna().sum():,}")

In [0]:
volume_path = CONFIG["volume_path"]
 
# Create Volume directory if it does not exist
dbutils.fs.mkdirs(volume_path)
 
# Remove existing files so we do not accumulate stale data on re-runs
print("Cleaning up existing files...")
for name in ["customers", "accounts", "transactions"]:
    try:
        dbutils.fs.rm(f"{volume_path}/{name}", recurse=True)
        print(f"  Removed {name}/")
    except:
        pass  # Does not exist yet — continue
 
def write_single_csv(df, name):
    """Write a pandas DataFrame as a single clean CSV to the Volume."""
    temp_path  = f"{volume_path}/temp_{name}"
    final_path = f"{volume_path}/{name}.csv"
 
    # Convert to Spark and write as single partition
    spark_df = spark.createDataFrame(df)
    spark_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(temp_path)
 
    # Find the part file Spark generated
    files     = dbutils.fs.ls(temp_path)
    part_file = [f.path for f in files if f.name.startswith("part-") and f.name.endswith(".csv")][0]
 
    # Rename to clean name and remove temp folder
    dbutils.fs.cp(part_file, final_path)
    dbutils.fs.rm(temp_path, recurse=True)
 
    print(f"  {name}.csv written ({len(df):,} rows)")
 
print("\nWriting CSV files to Volume...")
write_single_csv(customers,    "customers")
write_single_csv(accounts,     "accounts")
write_single_csv(transactions, "transactions")
 
print(f"\nAll files written to: {volume_path}")

In [0]:
print("=" * 60)
print("SANITY CHECK — Reading files back from Volume")
print("=" * 60)
 
for name in ["customers", "accounts", "transactions"]:
    df = spark.read.option("header", True).csv(f"{volume_path}/{name}.csv")
    null_counts = {
        col: df.filter(df[col].isNull()).count()
        for col in df.columns
        if df.filter(df[col].isNull()).count() > 0
    }
    print(f"\n[{name.upper()}]")
    print(f"  Row count    : {df.count():,}")
    print(f"  Column count : {len(df.columns)}")
    print(f"  Null columns : {null_counts}")
 
print("\nNotebook 01 complete. Raw CSVs ready for Bronze ingestion.")

In [0]:
import os

repo_sample_path = "/Workspace/Users/kamranhabib1212@gmail.com/datasentry/sample_data"

# Create folder if it doesn't exist
os.makedirs(repo_sample_path, exist_ok=True)

# Write 1,000-row samples using pandas directly
customers.head(1000).to_csv(f"{repo_sample_path}/customers_sample.csv",    index=False)
accounts.head(1000).to_csv(f"{repo_sample_path}/accounts_sample.csv",      index=False)
transactions.head(1000).to_csv(f"{repo_sample_path}/transactions_sample.csv", index=False)

print("Sample CSVs written to repo:")
print(f"  customers_sample.csv    (1,000 rows)")
print(f"  accounts_sample.csv     (1,000 rows)")
print(f"  transactions_sample.csv (1,000 rows)")
print(f"\nPath: {repo_sample_path}")
print("Commit and push from Databricks Repos to sync to GitHub.")